# Step 2. Vector DB: 벡터를 저장하고 빨리 찾기
Step 1에서는 벡터를 메모리에 두고 하나씩 비교했습니다. 이번에는 **Chroma(크로마)** 라는 Vector DB에 저장합니다.

## 핵심 개념
- **Vector DB**: 벡터를 저장해 두고, 질문 벡터와 가까운 것을 빨리 찾아 주는 DB
- **Chroma**: 설치만 하면 내 맥북의 폴더에 저장되는 가벼운 Vector DB. 서버를 띄울 필요가 없습니다.
- **Metadata(메타데이터)**: 벡터와 함께 저장하는 꼬리표(날짜, 분류). "이번 달 것만" 같은 조건 검색에 씁니다.

> AWS에서는 같은 일을 S3 Vectors, Aurora(pgvector), OpenSearch가 합니다.

In [ ]:
# 필요한 패키지를 설치합니다. 처음에는 1분쯤 걸립니다.
import subprocess, sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check", "ollama", "chromadb"], check=True)
print("✅ 패키지 준비 완료. 다음 셀로 넘어가세요.")

## 1. 설정

In [ ]:
import json

try:
    with open("config.json") as f:
        _config = json.load(f)
    print("✅ config.json 로드 완료")
except FileNotFoundError:
    raise SystemExit("❌ config.json이 없습니다. `0_check_setup` 노트북을 먼저 실행해 주세요.")

EMBED_MODEL = _config["EMBED_MODEL"]
LLM_MODEL = _config["LLM_MODEL"]
AGENT_MODEL = _config["AGENT_MODEL"]
print(f"   Embedding 모델: {EMBED_MODEL}")
print(f"   LLM 모델:       {LLM_MODEL}")
print(f"   Agent 모델:     {AGENT_MODEL}")

## 2. DB 만들기
이 노트북 옆의 `.lab-data` 폴더에 저장됩니다. 노트북을 껐다 켜도 남아 있습니다.

In [ ]:
import chromadb
import ollama

client = chromadb.PersistentClient(path=".lab-data/step2")
col = client.get_or_create_collection("notes", metadata={"hnsw:space": "cosine"})


def embed(text):
    return ollama.embed(model=EMBED_MODEL, input=text)["embeddings"][0]


print(f"✅ 컬렉션 준비 완료. 지금 들어 있는 문서: {col.count()}개")

## 3. 문서 넣기
문서, 벡터, 꼬리표를 함께 넣습니다. 같은 id로 다시 넣으면 덮어씁니다.

In [ ]:
# TODO: 내 메모를 추가해 보세요!
notes = [
    ("n1", "다음 주 화요일까지 RAG 과제 제출", "수업"),
    ("n2", "동아리 회식 장소는 학교 앞 삼겹살집", "생활"),
    ("n3", "병원 주차장은 지하 2층", "병원"),
    ("n4", "구내식당 점심은 11시 30분부터", "병원"),
    ("n5", "혈압약은 아침 식사 후에 복용", "건강"),
]

col.upsert(
    ids=[n[0] for n in notes],
    documents=[n[1] for n in notes],
    embeddings=[embed(n[1]) for n in notes],
    metadatas=[{"tag": n[2]} for n in notes],
)
print(f"✅ 저장 완료. 문서 수: {col.count()}개")

## 4. 의미로 찾기
`distance`는 거리입니다. **작을수록 가깝습니다.**

In [ ]:
def search(query, k=3, tag=None):
    r = col.query(
        query_embeddings=[embed(query)],
        n_results=k,
        where={"tag": tag} if tag else None,
    )
    print(f"🔎 질문: {query}" + (f"  (분류: {tag})" if tag else ""))
    for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0]):
        print(f"   거리 {dist:.3f}  [{meta['tag']}]  {doc}")
    print()


# TODO: 질문을 바꿔 보세요!
search("숙제 마감이 언제지?")
search("밥은 몇 시부터 먹을 수 있어?")

## 5. 꼬리표로 걸러서 찾기
"병원" 분류 안에서만 찾습니다. 의미 검색과 조건 검색을 함께 쓰는 것이 실무의 기본입니다.

In [ ]:
search("몇 시에 가면 돼?", tag="병원")
search("몇 시에 가면 돼?", tag="건강")

## 정리
- Vector DB는 "문서 + 벡터 + 꼬리표"를 저장하고, 가까운 것을 찾아 줍니다.
- 찾은 문서를 LLM에게 함께 보여 주면 **RAG**가 됩니다. → Step 3